# Week 5 – Text Classification
## Day 2: TF-IDF + Logistic Regression

**Objective:** Build and evaluate a multiclass
text classification model using TF-IDF
and Logistic Regression.

In [1]:
import pandas as pd

df = pd.read_csv("news_preprocessed.csv")

display(df.head())
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())  



,Text,Label,normalized_text
0,Budget to set scene for election\n \n Gordon B...,0,budget to set scene for election gordon brown ...
1,Army chiefs in regiments decision\n \n Militar...,0,army chiefs in regiments decision military chi...
2,Howard denies split over ID cards\n \n Michael...,0,howard denies split over id cards michael howa...
3,Observers to monitor UK election\n \n Minister...,0,observers to monitor uk election ministers wil...
4,Kilroy names election seat target\n \n Ex-chat...,0,kilroy names election seat target ex-chat show...


Dataset shape: (2127, 3)

Missing values:
Text               0
Label              0
normalized_text    0
dtype: int64


In [ ]:
# Drop duplicates based on the "normalized_text" column
df = df.drop_duplicates(subset=["normalized_text"]).copy()

X = df["normalized_text"]
y = df["Label"]

print("Final dataset shape:", df.shape)

Final dataset shape: (2124, 3)


In [3]:
print("Duplicate normalized texts:",
      df.duplicated(subset=["normalized_text"]).sum())

print("Empty normalized texts:",
      df["normalized_text"].str.strip().eq("").sum())



Duplicate normalized texts: 0
Empty normalized texts: 0


In [4]:
duplicate_texts = df[df.duplicated(subset=["normalized_text"], keep=False)]

display(duplicate_texts[["normalized_text", "Label"]])

,normalized_text,Label


In [5]:
### 3. Train/Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42, 
    stratify=y 
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1699
Testing samples: 425


In [51]:
pd.DataFrame(X_train)[pd.DataFrame(X_train)["normalized_text"].str.contains(r"call center" )]

,normalized_text


In [6]:
### 4. TF-IDF Feature Extraction

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (1699, 26849)
Testing TF-IDF shape: (425, 26849)


In [7]:
### 5. Logistic Regression Model Training

from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

In [8]:
### 6. Model Evaluation – Normalized Text

from sklearn.metrics import accuracy_score, f1_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9694117647058823
Macro F1: 0.9694486917783669
Weighted F1: 0.9693329324122765

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.93      0.96        81
           1       0.96      0.99      0.98       100
           2       0.97      0.97      0.97        69
           3       0.99      0.97      0.98        74
           4       0.95      0.98      0.97       101

    accuracy                           0.97       425
   macro avg       0.97      0.97      0.97       425
weighted avg       0.97      0.97      0.97       425



In [ ]:
# Original Text
samples = df.groupby("Label").sample(n=5, random_state=42).copy()

samples["Title"] = samples["Text"].str.strip().str.split("\n").str[0]

display(samples[["Label", "Title"]].sort_values("Label"))

,Label,Title
70,0,'EU referendum could cost Â£80m'
281,0,MSPs hear renewed climate warning
283,0,Tories 'would cut number of MPs'
33,0,Blair backs 'pre-election budget'
42,0,Labour MP praises Tory campaign
811,1,Thomas out of Six Nations
618,1,Sculthorpe wants Lions captaincy
466,1,Man Utd through after Exeter test
475,1,Wasps 31-37 Leicester
433,1,O'Connell rejects Lions rumours


Label	Category	Evidence from Articles
0	Politics	   EU referendum, Tories, Labour MP
1	Sports    	Man Utd, Six Nations, Leicester
2	Technology 	Cyber crime, Firefox, PC security
3	Entertainment   	Films, music, charity shows
4	Business	    Renault, consumer spending, energy companies

In [12]:
### 1. Extract Incorrect Predictions

results_df = pd.DataFrame({
    "Text": X_test,
    "Actual_Label": y_test,
    "Predicted_Label": y_pred
})

errors_df = results_df[
    results_df["Actual_Label"] != results_df["Predicted_Label"]
].copy()

print("Total incorrect predictions:", len(errors_df))

display(errors_df)

Total incorrect predictions: 13


,Text,Actual_Label,Predicted_Label
52,sport betting rules in spotlight a group of mp...,0,4
1818,call centre users 'lose patience' customers tr...,4,2
943,half-life 2 sweeps bafta awards pc first perso...,2,3
1296,musicians to tackle us red tape musicians' gro...,3,0
1341,'my memories of marley...' to mark the 60th an...,3,1
121,mp's shock at long lost daughter labour mp ste...,0,1
130,report attacks defence spending the ministry o...,0,4
1184,tech helps disabled speed demons an organisati...,2,1
243,'super union' merger plan touted two of britai...,0,4
2053,bt offers equal access to rivals bt has moved ...,4,2


In [13]:
### 3. Map Class Labels to Category Names

label_map = {
    0: "Politics",
    1: "Sports",
    2: "Technology",
    3: "Entertainment",
    4: "Business"
}

errors_df["Actual_Category"] = errors_df["Actual_Label"].map(label_map)
errors_df["Predicted_Category"] = errors_df["Predicted_Label"].map(label_map)

display(errors_df[["Actual_Category", "Predicted_Category"]])

,Actual_Category,Predicted_Category
52,Politics,Business
1818,Business,Technology
943,Technology,Entertainment
1296,Entertainment,Politics
1341,Entertainment,Sports
121,Politics,Sports
130,Politics,Business
1184,Technology,Sports
243,Politics,Business
2053,Business,Technology


In [45]:
#. Display the  Incorrect Prediction
errors_df["Original_Text"] = df.loc[errors_df.index, "Text"]

first_error = errors_df.iloc[1]

print("Actual Category:", first_error["Actual_Category"])
print("Predicted Category:", first_error["Predicted_Category"])

print("\nArticle Text:\n")
print(first_error["Original_Text"])

Actual Category: Business
Predicted Category: Technology

Article Text:

Call centre users 'lose patience'
 
 Customers trying to get through to call centres are getting impatient and quicker to hang up, a survey suggests.
 
 Once past the welcome message, callers on average hang up after just 65 seconds of listening to canned music. The drop in patience comes as the number of calls to call centres is growing at a rate of 20% every year. "Customers are getting used to the idea of an 'always available' society," says Cara Diemont of IT firm Dimension Data, which commissioned the survey. However, call centres also saw a sharp increase of customers simply abandoning calls, she says, from just over 5% in 2003 to a record 13.3% during last year. When automated phone message systems are taken out of the equation, where customers have to pick their way through multiple options and messages, the number of abandoned calls is even higher - a sixth of all callers give up rather than wait. One pos

In [ ]:
#  Error Analysis Report
error_report = errors_df[
    ["Actual_Label", "Predicted_Label",
     "Actual_Category", "Predicted_Category"]
].copy()

error_report["Article_Title"] = df.loc[
    error_report.index, "Text"
].str.split("\n").str[0].str.strip()

error_report["Possible_Cause"] = ""
error_report["Explanation"] = ""

error_report = error_report[
    ["Article_Title", "Actual_Label", "Predicted_Label",
     "Actual_Category", "Predicted_Category",
     "Possible_Cause", "Explanation"]
]

error_report.to_excel(
    "day3_error_analysis.xlsx",
    index=True,
    index_label="Article_ID"
)

print("Excel file updated successfully!")

Excel file updated successfully!


In [46]:
x_train

NameError: name 'x_train' is not defined

## Day 3: Error Analysis Verification
### Feature Contribution Analysis

In [44]:
### Feature Contribution Analysis
i = 1
error = errors_df.iloc[i]

actual = error["Actual_Label"]
predicted = error["Predicted_Label"]

features = vectorizer.get_feature_names_out()

tfidf_values = vectorizer.transform([error["Text"]]).toarray()[0]

contributions = tfidf_values * (
    model.coef_[predicted] - model.coef_[actual]
)

feature_analysis = pd.DataFrame({
    "Word": features,
    "Contribution": contributions
})

print("Actual:", label_map[actual])
print("Predicted:", label_map[predicted])

display(feature_analysis.nlargest(10, "Contribution"))

Actual: Business
Predicted: Technology


,Word,Contribution
17056,of,0.150973
26741,you,0.133224
2183,and,0.123120
24374,to,0.122058
2473,are,0.115940
17250,or,0.104389
13259,is,0.096242
24122,that,0.093221
24130,the,0.088379
6958,data,0.072834


### Verification Findings

The feature contribution analysis shows that
words such as "growth" and "betting" contributed
to the incorrect Business prediction.

These findings support the topic overlap hypothesis.
Additionally, some common words contributed to the
prediction, suggesting that stop-word removal
could be evaluated as a potential improvement.

In [37]:
verification_results = []

for article_id, error in errors_df.iterrows():

    actual = error["Actual_Label"]
    predicted = error["Predicted_Label"]

    tfidf_values = vectorizer.transform(
        [error["Text"]]
    ).toarray()[0]

    contributions = tfidf_values * (
        model.coef_[predicted] - model.coef_[actual]
    )

    feature_analysis = pd.DataFrame({
        "Word": features,
        "Contribution": contributions
    })

    top_words = feature_analysis[
        feature_analysis["Contribution"] > 0
    ].nlargest(10, "Contribution")

    verification_results.append({
        "Article_ID": article_id,
        "Actual_Category": label_map[actual],
        "Predicted_Category": label_map[predicted],
        "Top_Contributing_Words": ", ".join(top_words["Word"])
    })

verification_df = pd.DataFrame(verification_results)

display(verification_df)

,Article_ID,Actual_Category,Predicted_Category,Top_Contributing_Words
0,52,Politics,Business,"growth, betting, group, corruption, in, us, ha..."
1,1818,Business,Technology,"of, you, and, to, are, or, is, that, the, data"
2,943,Technology,Entertainment,"award, best, awards, won, was, music, year, ca..."
3,1296,Entertainment,Politics,"to, the, mr, are, said, union, secretary, we, ..."
4,1341,Entertainment,Sports,"he, to, cup, but, him, we, world, football, fi..."
5,121,Politics,Sports,"her, his, my, at, pound, when, first, out, we,..."
6,130,Politics,Business,"projects, spending, its, total, year, costs, m..."
7,1184,Technology,Sports,"his, she, her, club, he, in, left, me, we, at"
8,243,Politics,Business,"merger, executives, trade, bid, in, has, agree..."
9,2053,Business,Technology,"bt, broadband, to, the, of, and, access, servi..."


## Model 2: Full Clean Text Classification

### Objective
Evaluate the impact of advanced text preprocessing
on classification performance while maintaining
the same algorithm and train-test split.



In [ ]:
### 1. Load the Fully Preprocessed Dataset
df_full = pd.read_csv("news_full_preprocessed.csv")

print("Dataset shape:", df_full.shape)

display(df_full.head())

Dataset shape: (2127, 4)


,Text,Label,normalized_text,full_clean_text
0,Budget to set scene for election\n \n Gordon B...,0,budget to set scene for election gordon brown ...,budget set scene election gordon brown seek ec...
1,Army chiefs in regiments decision\n \n Militar...,0,army chiefs in regiments decision military chi...,army chiefs regiment decision military chief e...
2,Howard denies split over ID cards\n \n Michael...,0,howard denies split over id cards michael howa...,howard deny split d card michael howard deny s...
3,Observers to monitor UK election\n \n Minister...,0,observers to monitor uk election ministers wil...,observer monitor uk election minister invite i...
4,Kilroy names election seat target\n \n Ex-chat...,0,kilroy names election seat target ex-chat show...,kilroy name election seat target ex - chat hos...


In [ ]:
#2. Drop duplicates based on the "normalized_text" column
df_full = df_full.drop_duplicates(
    subset=["normalized_text"]
).copy()

print("Dataset shape:", df_full.shape)

Dataset shape: (2124, 4)


In [31]:
### 3. Verify Dataset Consistency 
print("Same indices:", df.index.equals(df_full.index))

print("Same texts:", df["normalized_text"].equals(
    df_full["normalized_text"]
))

print("Same labels:", df["Label"].equals(
    df_full["Label"]
))


Same indices: True
Same texts: True
Same labels: True


In [30]:
### 4. Reuse the Original Train-Test Split
X_train_full = df_full.loc[X_train.index, "full_clean_text"]

X_test_full = df_full.loc[X_test.index, "full_clean_text"]

print("Training samples:", X_train_full.shape)
print("Testing samples:", X_test_full.shape)

Training samples: (1699,)
Testing samples: (425,)


In [32]:
### 5. TF-IDF Feature Extraction
vectorizer_full = TfidfVectorizer()

X_train_full_tfidf = vectorizer_full.fit_transform(X_train_full)

X_test_full_tfidf = vectorizer_full.transform(X_test_full)

print("Training shape:", X_train_full_tfidf.shape)
print("Testing shape:", X_test_full_tfidf.shape)

Training shape: (1699, 21197)
Testing shape: (425, 21197)


In [33]:
### 6. Train Logistic Regression Model
model_full = LogisticRegression(max_iter=1000)

model_full.fit(X_train_full_tfidf, y_train)

y_pred_full = model_full.predict(X_test_full_tfidf)

In [34]:
### 7. Evaluate Model Performance
accuracy_full = accuracy_score(y_test, y_pred_full)

print("Accuracy:", accuracy_full)

print(classification_report(y_test, y_pred_full))

Accuracy: 0.9647058823529412
              precision    recall  f1-score   support

           0       0.96      0.95      0.96        81
           1       0.97      0.99      0.98       100
           2       0.98      0.90      0.94        69
           3       0.95      0.99      0.97        74
           4       0.96      0.98      0.97       101

    accuracy                           0.96       425
   macro avg       0.97      0.96      0.96       425
weighted avg       0.97      0.96      0.96       425



In [35]:
### 8. Model Performance Comparison
from sklearn.metrics import f1_score

comparison_df = pd.DataFrame({
    "Model": ["Normalized Text", "Full Clean Text"],

    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_full)
    ],

    "Macro F1": [
        f1_score(y_test, y_pred, average="macro"),
        f1_score(y_test, y_pred_full, average="macro")
    ],

    "Weighted F1": [
        f1_score(y_test, y_pred, average="weighted"),
        f1_score(y_test, y_pred_full, average="weighted")
    ],

    "Incorrect Predictions": [
        (y_test != y_pred).sum(),
        (y_test != y_pred_full).sum()
    ]
})

display(comparison_df.round(4))

,Model,Accuracy,Macro F1,Weighted F1,Incorrect Predictions
0,Normalized Text,0.9694,0.9694,0.9693,13
1,Full Clean Text,0.9647,0.9627,0.9645,15


In [36]:
baseline_wrong = y_test != y_pred

full_wrong = y_test != y_pred_full

corrected_errors = baseline_wrong & ~full_wrong

new_errors = ~baseline_wrong & full_wrong

persistent_errors = baseline_wrong & full_wrong

both_correct = ~baseline_wrong & ~full_wrong

print("Corrected Errors:", corrected_errors.sum())
print("New Errors:", new_errors.sum())
print("Persistent Errors:", persistent_errors.sum())
print("Both Correct:", both_correct.sum())

Corrected Errors: 5
New Errors: 7
Persistent Errors: 8
Both Correct: 405


In [38]:
### 10. Analyze Corrected and New Errors
error_comparison = pd.DataFrame({
    "Actual": y_test,
    "Baseline_Prediction": y_pred,
    "Full_Clean_Prediction": y_pred_full
})

error_comparison = error_comparison.apply(
    lambda column: column.map(label_map)
)

print("Corrected Errors:")
display(error_comparison[corrected_errors])

print("New Errors:")
display(error_comparison[new_errors])

print("Persistent Errors:")
display(error_comparison[persistent_errors])


Corrected Errors:


,Actual,Baseline_Prediction,Full_Clean_Prediction
1818,Business,Technology,Business
1341,Entertainment,Sports,Entertainment
121,Politics,Sports,Politics
1184,Technology,Sports,Technology
276,Politics,Business,Politics


New Errors:


,Actual,Baseline_Prediction,Full_Clean_Prediction
933,Technology,Technology,Sports
1105,Technology,Technology,Sports
982,Technology,Technology,Politics
1232,Technology,Technology,Entertainment
1150,Technology,Technology,Entertainment
1683,Business,Business,Politics
1136,Technology,Technology,Entertainment


Persistent Errors:


,Actual,Baseline_Prediction,Full_Clean_Prediction
52,Politics,Business,Business
943,Technology,Entertainment,Entertainment
1296,Entertainment,Politics,Politics
130,Politics,Business,Business
243,Politics,Business,Business
2053,Business,Technology,Technology
166,Politics,Sports,Sports
795,Sports,Business,Business


### 11. Final Findings

- The Normalized Text model achieved 96.94% accuracy.
- The Full Clean Text model achieved 96.47% accuracy.
- Full text cleaning corrected 5 original errors but
  introduced 7 new errors, while 8 errors persisted.
- Advanced preprocessing does not necessarily improve
  classification performance.
- Removing stop words and applying lemmatization may
  remove useful information.

### Model Selection

The Normalized Text model was selected because it
achieved higher accuracy, Macro F1, and Weighted F1
on the current test set.